In [162]:
import numpy as np
import pandas as pd

## Import Dataset and analysis

In [163]:
df = pd.read_csv('cars.csv')

In [164]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   brand          8128 non-null   object
 1   km_driven      8128 non-null   int64 
 2   fuel           8128 non-null   object
 3   owner          8128 non-null   object
 4   selling_price  8128 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 317.6+ KB


In [165]:
df.sample(5)

,brand,km_driven,fuel,owner,selling_price
3423,BMW,8500,Diesel,First Owner,5500000
736,Toyota,50000,Diesel,First Owner,750000
1032,Hyundai,60000,Petrol,First Owner,400000
4932,Maruti,59500,Petrol,Second Owner,421000
5945,Mahindra,40200,Diesel,First Owner,445000


In [166]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [167]:
df['fuel'].value_counts()

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [168]:
df['brand'].value_counts()

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Force               6
Land                6
Isuzu               5
Kia                 4
Ambassador          4
Daewoo              3
MG                  3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

## Thresholding and filtering brands

In [169]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [170]:
df.sample(10) # Before count mapping

,brand,km_driven,fuel,owner,selling_price
6201,Volkswagen,120000,Diesel,Second Owner,550000
554,Hyundai,32000,Petrol,First Owner,580000
697,Mahindra,167000,Diesel,Second Owner,350000
6170,Volvo,20000,Diesel,First Owner,3800000
4799,BMW,30000,Diesel,First Owner,2150000
6497,Hyundai,50000,Diesel,First Owner,1000000
483,Maruti,80000,Diesel,First Owner,750000
1795,Chevrolet,88000,Diesel,Third Owner,290000
1053,Ford,30000,Diesel,First Owner,793000
1955,Maruti,40000,Petrol,First Owner,225000


In [171]:
df.sample(10,random_state=110) # AFter count mapping

,brand,km_driven,fuel,owner,selling_price
4063,Hyundai,25000,Petrol,First Owner,425000
3212,Toyota,80000,Diesel,First Owner,950000
705,Hyundai,17000,Petrol,First Owner,290000
2520,Maruti,60000,Diesel,First Owner,610000
2135,Audi,70000,Diesel,First Owner,1850000
4610,Honda,35000,Petrol,First Owner,894999
669,Fiat,90000,Diesel,First Owner,400000
5056,Maruti,80000,Petrol,Second Owner,120000
2580,Maruti,100000,Diesel,First Owner,405000
1514,Honda,92000,Diesel,First Owner,900000


In [172]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:-1], df['selling_price'],random_state=11)

In [173]:
X_train

,brand,km_driven,fuel,owner
4711,Hyundai,70000,Diesel,Second Owner
6626,Mahindra,91000,Diesel,First Owner
1744,Volkswagen,52000,Diesel,First Owner
789,Tata,120000,Diesel,First Owner
220,Maruti,25000,Petrol,Second Owner
...,...,...,...,...
7505,Mahindra,30000,Diesel,First Owner
7259,Mahindra,200000,Diesel,First Owner
5200,Toyota,90000,Diesel,First Owner
3775,Toyota,30000,Diesel,First Owner


In [175]:
threshold = 100
counts = X_train['brand'].value_counts()

X_train['brand'] = np.where(X_train['brand'].map(counts)>=threshold,X_train['brand'],'Others')
X_test['brand'] = np.where(X_test['brand'].map(counts)>=threshold,X_test['brand'],'Others')


In [176]:
OHE = OneHotEncoder(sparse_output=False,dtype = np.int32,handle_unknown='ignore',drop='first')

In [177]:
X_train_encoded = OHE.fit_transform(X_train[['brand']])

In [178]:
cat_brand = OHE.categories_[0]

In [201]:
cat_brand

array(['Chevrolet', 'Ford', 'Honda', 'Hyundai', 'Mahindra', 'Maruti',
       'Others', 'Renault', 'Tata', 'Toyota', 'Volkswagen'], dtype=object)

In [193]:
X_train.index

Index([4711, 6626, 1744,  789,  220, 2606, 3372, 7146, 4194, 7148,
       ...
        583, 5793,  332, 1293, 4023, 7505, 7259, 5200, 3775, 1945],
      dtype='int64', length=6096)

In [186]:
X_train_encoded

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       ...,
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(6096, 10), dtype=int32)

In [180]:
X_train = X_train.drop('brand',axis=1)

In [205]:
new_X_train = pd.concat([X_train, pd.DataFrame(X_train_encoded, columns = cat_brand[1:], index=X_train.index)], axis=1)
new_X_train

,km_driven,fuel,owner,Ford,Honda,Hyundai,Mahindra,Maruti,Others,Renault,Tata,Toyota,Volkswagen
4711,70000,Diesel,Second Owner,0,0,1,0,0,0,0,0,0,0
6626,91000,Diesel,First Owner,0,0,0,1,0,0,0,0,0,0
1744,52000,Diesel,First Owner,0,0,0,0,0,0,0,0,0,1
789,120000,Diesel,First Owner,0,0,0,0,0,0,0,1,0,0
220,25000,Petrol,Second Owner,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7505,30000,Diesel,First Owner,0,0,0,1,0,0,0,0,0,0
7259,200000,Diesel,First Owner,0,0,0,1,0,0,0,0,0,0
5200,90000,Diesel,First Owner,0,0,0,0,0,0,0,0,1,0
3775,30000,Diesel,First Owner,0,0,0,0,0,0,0,0,1,0


In [204]:
df.iloc[4711]

brand                 Hyundai
km_driven               70000
fuel                   Diesel
owner            Second Owner
selling_price          320000
Name: 4711, dtype: object

In [216]:
new_X_train.loc[4711]

km_driven            70000
fuel                Diesel
owner         Second Owner
Ford                     0
Honda                    0
Hyundai                  1
Mahindra                 0
Maruti                   0
Others                   0
Renault                  0
Tata                     0
Toyota                   0
Volkswagen               0
Name: 4711, dtype: object

# pd get dummies

In [ ]:
df.head()


In [ ]:
df.shape

In [ ]:
pd.get_dummies(df[['fuel','owner']],dtype=np.int32)

In [ ]:
fuel_owner  = pd.get_dummies(df[['fuel','owner']],drop_first=True,dtype=np.int32)
fuel_owner

In [ ]:
repl = counts[counts <= threshold].index
(df['brand'].replace(repl, 'uncommon'))

In [ ]:
df

In [ ]:
pd.concat([df.drop(['fuel','owner'],axis=1),fuel_owner])